# Para / Perp Summary Plots

Plot compact para/perp ablation summaries with cleaner labels and tighter styling.

In [ ]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Match the EMNLP/ACL paper style, which uses the Times text font.
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'Nimbus Roman No9 L', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'analysis' / 'visualization' / 'para_ablation').is_dir() and (candidate / 'results' / 'figures').is_dir():
            return candidate
    raise RuntimeError(f'Could not locate project root from {start}')


PAPER_ROOT = find_project_root()
NOTEBOOK_DIR = PAPER_ROOT / 'analysis' / 'visualization' / 'para_ablation'
DATA_DIR = NOTEBOOK_DIR / 'data'

PARA_INPUT_TSV = DATA_DIR / 'para_ppl_summary.tsv'
PERP_INPUT_TSV = DATA_DIR / 'perp_ppl_summary.tsv'
PARA_OUTPUT_DIR = PAPER_ROOT / 'results' / 'figures' / 'para_ablation'
PERP_OUTPUT_DIR = PARA_OUTPUT_DIR

# Optional raw compare dirs for variance computation from per-sample JSONL.
# If empty/not found, plotting falls back to mean curves only.
PARA_COMPARE_DIR = Path('')
PERP_COMPARE_DIR = Path('')

PARA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PERP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

para_df = pd.read_csv(PARA_INPUT_TSV, sep='	').sort_values('para_scale').reset_index(drop=True)
perp_df = pd.read_csv(PERP_INPUT_TSV, sep='	').sort_values('perp_scale').reset_index(drop=True)

# column compatibility
COL_MEAN_XSA = 'mean_per_text_delta_ppl_xsa' if 'mean_per_text_delta_ppl_xsa' in para_df.columns else 'weighted_delta_loss_xsa'
COL_MEAN_BASELINE = 'mean_per_text_baseline_ppl' if 'mean_per_text_baseline_ppl' in para_df.columns else 'weighted_baseline_ppl'
COL_MEAN_RESIDUAL_ATTN = 'mean_per_text_residual_attn_ppl' if 'mean_per_text_residual_attn_ppl' in para_df.columns else 'weighted_residual_attn_ppl'
COL_MEAN_RESIDUAL_MLP = 'mean_per_text_residual_mlp_ppl' if 'mean_per_text_residual_mlp_ppl' in para_df.columns else 'weighted_residual_mlp_ppl'
COL_MEAN_RESIDUAL_BOTH = 'mean_per_text_residual_both_ppl' if 'mean_per_text_residual_both_ppl' in para_df.columns else 'weighted_residual_both_ppl'
C_XSA = '#1f77b4'
C_RA = '#ff7f0e'
C_RM = '#2ca02c'
C_REF = '#7f7f7f'
REMOVE_VALUE_BASED_POINTS = False
SMOOTH_LINES = False

SMOOTH_WINDOW = 21
SMOOTH_POINTS = 300
LOG_LINTHRESH_LOSS = 0.05
LOG_LINTHRESH_PPL = 0.05


In [ ]:
def _style_axis(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, axis='y')


def _legend_1x3(ax, loc='best', *, ncol=3, fontsize=16.0, bbox_to_anchor=None):
    handles, labels = ax.get_legend_handles_labels()
    wanted = ['Value-Based', 'Residual(Attn)', 'Residual(MLP)']
    legend_kw = dict(
        loc=loc,
        ncol=ncol,
        frameon=True,
        fancybox=True,
        framealpha=1.0,
        facecolor='white',
        edgecolor='#cfcfcf',
        columnspacing=1.25,
        handletextpad=0.65,
        borderpad=0.42,
        fontsize=fontsize,
        borderaxespad=0.4,
    )
    if bbox_to_anchor is not None:
        legend_kw['bbox_to_anchor'] = bbox_to_anchor
    if set(labels) == set(wanted) and len(labels) == 3:
        by_label = {label: handle for handle, label in zip(handles, labels)}
        ordered_labels = ['Value-Based', 'Residual(Attn)', 'Residual(MLP)']
        ordered_handles = [by_label[label] for label in ordered_labels]
        ax.legend(ordered_handles, ordered_labels, **legend_kw)
    else:
        ax.legend(**legend_kw)


def _extract_std_from_jsonl(compare_dir: Path, mode: str):
    """Return dict[x_value] -> std(delta_ppl_xsa) by reading compare JSONL files."""
    if not compare_dir or not str(compare_dir) or not compare_dir.exists():
        return {}

    out = {}
    if mode == 'para':
        pat = re.compile(r'compare_para_scale_([-+]?\d+(?:p\d+)?(?:\.\d+)?)\.jsonl$')
        files = sorted(compare_dir.glob('compare_para_scale_*.jsonl'))
    else:
        pat = re.compile(r'compare_para_scale_[-+]?\d+(?:p\d+)?(?:\.\d+)?_perp_scale_([-+]?\d+(?:p\d+)?(?:\.\d+)?)\.jsonl$')
        files = sorted(compare_dir.glob('compare_para_scale_*_perp_scale_*.jsonl'))

    for fp in files:
        m = pat.search(fp.name)
        if not m:
            continue
        raw = m.group(1).replace('p', '.')
        try:
            xval = float(raw)
        except ValueError:
            continue
        vals = []
        with fp.open('r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                if 'delta_ppl' in obj and isinstance(obj['delta_ppl'], (int, float)):
                    vals.append(float(obj['delta_ppl']))
                elif 'mean_per_text_delta_ppl_xsa' in obj and isinstance(obj['mean_per_text_delta_ppl_xsa'], (int, float)):
                    vals.append(float(obj['mean_per_text_delta_ppl_xsa']))
        if vals:
            out[xval] = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    return out


def _variance_band(df, x_col, y_col, std_map):
    if not std_map:
        return None, None, None
    xs = df[x_col].to_numpy(dtype=float)
    ys = df[y_col].to_numpy(dtype=float)
    std = np.array([std_map.get(float(x), np.nan) for x in xs], dtype=float)
    if np.isnan(std).all():
        return None, None, None
    return xs, ys - std, ys + std


def _smooth_xy(x, y, num_points=300, window=21):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 4 or len(y) < 4 or not SMOOTH_LINES:
        return x, y
    order = np.argsort(x)
    x = x[order]
    y = y[order]
    x_dense = np.linspace(x.min(), x.max(), num_points)
    y_dense = np.interp(x_dense, x, y)
    win = int(window)
    if win < 3:
        return x_dense, y_dense
    if win % 2 == 0:
        win += 1
    kernel = np.hanning(win)
    if kernel.sum() == 0:
        return x_dense, y_dense
    kernel = kernel / kernel.sum()
    pad = win // 2
    y_pad = np.pad(y_dense, pad_width=pad, mode='edge')
    y_smooth = np.convolve(y_pad, kernel, mode='valid')
    return x_dense, y_smooth


def _plot_series(ax, x, y, *, label, color, lw, alpha, ms, marker, ls, smooth_window=None, show_points=True, point_alpha=None):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if SMOOTH_LINES and len(x) >= 4:
        xs, ys = _smooth_xy(x, y, num_points=SMOOTH_POINTS, window=(SMOOTH_WINDOW if smooth_window is None else smooth_window))
        ax.plot(xs, ys, label=label, color=color, linewidth=lw, alpha=alpha, linestyle=ls)
        if show_points:
            pa = min(0.95, alpha + 0.02) if point_alpha is None else point_alpha
            ax.plot(x, y, linestyle='None', marker=marker, markersize=ms, color=color, alpha=pa)
    else:
        ax.plot(x, y, label=label, color=color, linewidth=lw, alpha=alpha, marker=marker if show_points else None, markersize=ms, linestyle=ls)


def plot_overview(df, x_col, out_path, title_prefix, x_label, std_map=None):
    fig, ax = plt.subplots(figsize=(9.6, 5.0))

    x = df[x_col].to_numpy(dtype=float)
    _plot_series(ax, x, df[COL_MEAN_XSA].to_numpy(dtype=float), label='XSA mean', color=C_XSA, lw=2.3, alpha=1.0, ms=6.0, marker='o', ls='-')
    _plot_series(ax, x, df[COL_MEAN_RA].to_numpy(dtype=float), label='Residual attn mean', color=C_RA, lw=1.9, alpha=1.0, ms=5.0, marker='o', ls='-')
    _plot_series(ax, x, df[COL_MEAN_RM].to_numpy(dtype=float), label='Residual MLP mean', color=C_RM, lw=1.9, alpha=1.0, ms=5.0, marker='o', ls='-')

    xs, low, high = _variance_band(df, x_col, COL_MEAN_XSA, std_map or {})
    if xs is not None:
        ax.fill_between(xs, low, high, color=C_XSA, alpha=0.16, linewidth=0, label='XSA ±1 std')

    ax.axhline(0.0, color=C_REF, linestyle='--', linewidth=1.0)
    ax.set_yscale('symlog', linthresh=1.0)
    ax.set_xlabel(x_label)
    ax.set_ylabel('$\Delta \mathrm{PPL}$')
    _style_axis(ax)
    ax.legend(loc='upper right', ncol=3, frameon=True, fancybox=True, framealpha=1.0, facecolor='white', edgecolor='#cfcfcf')

    fig.savefig(out_path, dpi=300)
    plt.show()
    plt.close(fig)
    return out_path


def plot_xsa_zoom(df, x_col, out_path, title_prefix, x_label, std_map=None):
    fig, ax = plt.subplots(figsize=(8.8, 4.4))

    x = df[x_col].to_numpy(dtype=float)
    _plot_series(ax, x, df[COL_MEAN_XSA].to_numpy(dtype=float), label='XSA mean', color=C_XSA, lw=2.4, alpha=1.0, ms=6.0, marker='o', ls='-')
    xs, low, high = _variance_band(df, x_col, COL_MEAN_XSA, std_map or {})
    if xs is not None:
        ax.fill_between(xs, low, high, color=C_XSA, alpha=0.18, linewidth=0, label='XSA ±1 std')

    ax.axhline(0.0, color=C_REF, linestyle='--', linewidth=1.0)
    ax.axvline(1.0, color=C_REF, linestyle=':', linewidth=1.0)

    ax.set_xlabel(x_label)
    ax.set_ylabel('$\Delta \mathrm{PPL}$')
    _style_axis(ax)
    ax.legend(loc='best', frameon=True, fancybox=False, edgecolor='#666666')

    fig.savefig(out_path, dpi=300)
    plt.show()
    plt.close(fig)
    return out_path




def _resolve_weighted_delta_ppl(df, col):
    if col in df.columns:
        return df[col].to_numpy(dtype=float)
    if not col.startswith('weighted_delta_ppl_'):
        return None
    suffix = col[len('weighted_delta_ppl_'):]
    candidate = f'weighted_{suffix}_ppl'
    baseline = 'weighted_baseline_ppl'
    if candidate in df.columns and baseline in df.columns:
        return df[candidate].to_numpy(dtype=float) - df[baseline].to_numpy(dtype=float)
    return None


def plot_delta_loss(df, x_col, series, out_path, x_label):
    fig, ax = plt.subplots(figsize=(9.6, 5.0))
    x = df[x_col].to_numpy(dtype=float)
    ys_all = []
    for label, col, color, lw, alpha, ms, marker, ls in series:
        if col not in df.columns:
            continue
        y = df[col].to_numpy(dtype=float)
        ys_all.append(y)
        show_points = not (REMOVE_VALUE_BASED_POINTS and label == 'Value-Based')
        smooth_window = max(SMOOTH_WINDOW, 31) if (SMOOTH_LINES and label == 'Value-Based') else SMOOTH_WINDOW
        _plot_series(ax, x, y, label=label, color=color, lw=lw, alpha=alpha, ms=ms, marker=marker, ls=ls, smooth_window=smooth_window, show_points=show_points, point_alpha=0.45 if show_points else None)
    ax.axhline(0.0, color=C_REF, linestyle='--', linewidth=1.0)
    positive_vals = []
    for arr in ys_all:
        arr = np.asarray(arr, dtype=float)
        arr = arr[np.isfinite(arr) & (arr > 0)]
        if arr.size:
            positive_vals.append(arr)
    if positive_vals:
        all_pos = np.concatenate(positive_vals)
        if all_pos.max() / max(all_pos.min(), 1e-12) > 20:
            ax.set_yscale('symlog', linthresh=LOG_LINTHRESH_LOSS)
            max_exp = int(np.ceil(np.log10(all_pos.max())))
            min_exp = int(np.floor(np.log10(max(all_pos.min(), LOG_LINTHRESH_LOSS))))
            exp_values = list(range(min_exp, max_exp + 1))
            if len(exp_values) > 6:
                exp_values = exp_values[::2]
                if exp_values[-1] != max_exp:
                    exp_values.append(max_exp)
            ticks = [0.0] + [10.0**e for e in exp_values]
            ticks = sorted(set(ticks))
            ax.set_yticks(ticks)
            ax.set_yticklabels(_positive_log_tick_labels(ticks))
        else:
            _maybe_set_nonnegative_ylim(ax, ys_all)
    else:
        _maybe_set_nonnegative_ylim(ax, ys_all)
    ax.set_xlabel(x_label, fontsize=20)
    ax.set_ylabel('$\Delta \ell_{w}$', fontsize=20)
    ax.tick_params(axis='both', which='major', labelsize=17, length=5.4, width=1.05)
    _style_axis(ax)
    _legend_1x3(ax, loc='best', fontsize=16.0)
    fig.savefig(out_path, dpi=300)
    plt.show()
    plt.close(fig)
    return out_path

def _positive_log_tick_labels(ticks):
    labels = []
    for t in ticks:
        if abs(t) < 1e-12:
            labels.append('0')
        elif t > 0 and t < 1:
            labels.append(f'{t:g}')
        elif t > 0:
            labels.append(rf'$10^{{{int(round(np.log10(t)))}}}$')
        else:
            labels.append('0')
    return labels


def _symlog_tick_labels(ticks):
    labels = []
    for t in ticks:
        if abs(t) < 1e-12:
            labels.append('0')
        elif t > 0:
            if t < 1:
                labels.append(f'{t:g}')
            else:
                labels.append(rf'$10^{{{int(round(np.log10(t)))}}}$')
        else:
            if abs(t) < 1:
                labels.append(f'{t:g}')
            else:
                labels.append(rf'$-10^{{{int(round(np.log10(abs(t))))}}}$')
    return labels


def plot_delta_ppl_signed_log(
    df,
    x_col,
    series,
    out_path,
    x_label,
    *,
    fig_size=None,
    legend_loc='upper right',
    legend_ncol=1,
    legend_fontsize=None,
    legend_bbox_to_anchor=None,
    x_pad_ratio=None,
):
    fig_size = fig_size or ((7.55, 5.1) if x_col == 'perp_scale' else (9.5, 5.2))
    fig, ax = plt.subplots(figsize=fig_size)
    x = df[x_col].to_numpy(dtype=float)
    plotted = 0
    y_arrays = []
    for label, col, color, lw, alpha, ms, marker, ls in series:
        y = _resolve_weighted_delta_ppl(df, col)
        if y is None:
            continue
        y = np.asarray(y, dtype=float)
        y_arrays.append(y)
        show_points = not (REMOVE_VALUE_BASED_POINTS and label == 'Value-Based')
        smooth_window = max(SMOOTH_WINDOW, 31) if (SMOOTH_LINES and label == 'Value-Based') else SMOOTH_WINDOW
        _plot_series(
            ax,
            x,
            y,
            label=label,
            color=color,
            lw=lw,
            alpha=alpha,
            ms=ms,
            marker=marker,
            ls=ls,
            smooth_window=smooth_window,
            show_points=show_points,
            point_alpha=0.45 if show_points else None,
        )
        plotted += 1

    ax.axhline(0.0, color=C_REF, linestyle='--', linewidth=1.0)
    ax.set_yscale('symlog', linthresh=LOG_LINTHRESH_PPL)

    if y_arrays:
        vals = np.concatenate([np.ravel(np.asarray(v, dtype=float)) for v in y_arrays])
        vals = vals[np.isfinite(vals)]
        if vals.size:
            ymin = float(vals.min())
            ymax = float(vals.max())
            top = ymax * (1.035 if x_col == 'perp_scale' else 1.04) if ymax > 0 else 1.0
            if ymin >= 0:
                bottom = 0.0
            else:
                neg_ratio = abs(ymin) / max(ymax, LOG_LINTHRESH_PPL)
                if neg_ratio <= 0.02:
                    bottom = max(ymin * 1.08, -1.25)
                elif neg_ratio <= 0.08:
                    bottom = max(ymin * 1.1, -3.0)
                else:
                    bottom = ymin * 1.03
                if x_col == 'perp_scale':
                    bottom = max(bottom, ymin * 1.02)
            ax.set_ylim(bottom=bottom, top=top)

            pos_ticks = []
            if ymax > 0:
                max_pos_exp = int(np.ceil(np.log10(max(ymax, 1.0))))
                pos_exps = list(range(0, max_pos_exp + 1))
                if len(pos_exps) > 5:
                    pos_exps = pos_exps[::2]
                    if pos_exps[-1] != max_pos_exp:
                        pos_exps.append(max_pos_exp)
                pos_ticks = [10.0 ** e for e in pos_exps]

            neg_ticks = []
            if ymin < 0:
                if abs(ymin) < 1.0:
                    neg_tick_mag = 0.5 if abs(ymin) <= 0.75 else 1.0
                    neg_ticks = [-neg_tick_mag] if ymin <= -0.08 else []
                else:
                    max_neg_exp = int(np.ceil(np.log10(abs(ymin))))
                    neg_ticks = [-(10.0 ** e) for e in range(0, max_neg_exp + 1, 2)]
                    if not neg_ticks or abs(neg_ticks[-1]) < abs(ymin):
                        neg_ticks.append(-(10.0 ** max_neg_exp))

            ticks = sorted(set([0.0] + pos_ticks + neg_ticks))
            ax.set_yticks(ticks)
            ax.set_yticklabels(_symlog_tick_labels(ticks))

    xmin = float(np.min(x))
    xmax = float(np.max(x))
    xr = xmax - xmin
    pad = x_pad_ratio if x_pad_ratio is not None else (0.006 if x_col == 'perp_scale' else 0.018)
    pad = pad * xr
    ax.set_xlim(xmin - pad, xmax + pad)
    ax.set_xlabel(x_label, fontsize=20)
    ax.set_ylabel('$\Delta \mathrm{PPL}$', fontsize=20)
    ax.tick_params(axis='both', which='major', labelsize=17, length=5.4, width=1.05)
    _style_axis(ax)
    if plotted > 0:
        _legend_1x3(
            ax,
            loc=legend_loc,
            ncol=legend_ncol,
            fontsize=(legend_fontsize if legend_fontsize is not None else 16.0),
            bbox_to_anchor=legend_bbox_to_anchor,
        )
    fig.tight_layout()
    fig.savefig(out_path, dpi=300)
    plt.show()
    plt.close(fig)
    return out_path


## Para. Scale

In [ ]:
plot_delta_loss(
    para_df,
    x_col='para_scale',
    series=[
        ('Value-Based', 'weighted_delta_loss_xsa', C_XSA, 3.4, 1.0, 7.5, 'o', '-'),
        ('Residual(Attn)', 'weighted_delta_loss_residual_attn', C_RA, 2.6, 0.95, 6.4, 's', '--'),
        ('Residual(MLP)', 'weighted_delta_loss_residual_mlp', C_RM, 2.6, 0.95, 6.4, '^', '-.'),
    ],
    out_path=PARA_OUTPUT_DIR / 'para_scale_weighted_delta_loss.png',
    x_label='Para. Scale',
)

In [ ]:
plot_delta_ppl_signed_log(
    para_df,
    x_col='para_scale',
    series=[
        ('Value-Based', 'weighted_delta_ppl_xsa', C_XSA, 3.4, 1.0, 7.5, 'o', '-'),
        ('Residual(Attn)', 'weighted_delta_ppl_residual_attn', C_RA, 2.6, 0.95, 6.4, 's', '--'),
        ('Residual(MLP)', 'weighted_delta_ppl_residual_mlp', C_RM, 2.6, 0.95, 6.4, '^', '-.'),
    ],
    out_path=PARA_OUTPUT_DIR / 'para_scale_weighted_delta_ppl.pdf',
    x_label='Para. Scale',
    fig_size=(8.4, 5.2),
    legend_loc='upper right',
    legend_ncol=1,
    legend_fontsize=16.0,
    legend_bbox_to_anchor=(0.382, 0.555),
    x_pad_ratio=0.018,
)


## Perp. Scale

In [ ]:
plot_delta_loss(
    perp_df,
    x_col='perp_scale',
    series=[
        ('Value-Based', 'weighted_delta_loss_xsa', C_XSA, 3.4, 1.0, 7.5, 'o', '-'),
        ('Residual(Attn)', 'weighted_delta_loss_residual_attn', C_RA, 2.6, 0.95, 6.4, 's', '--'),
        ('Residual(MLP)', 'weighted_delta_loss_residual_mlp', C_RM, 2.6, 0.95, 6.4, '^', '-.'),
    ],
    out_path=PERP_OUTPUT_DIR / 'perp_scale_weighted_delta_loss.png',
    x_label='Perp. Scale',
)

In [ ]:
plot_delta_ppl_signed_log(
    perp_df,
    x_col='perp_scale',
    series=[
        ('Value-Based', 'weighted_delta_ppl_xsa', C_XSA, 3.4, 1.0, 7.5, 'o', '-'),
        ('Residual(Attn)', 'weighted_delta_ppl_residual_attn', C_RA, 2.6, 0.95, 6.4, 's', '--'),
        ('Residual(MLP)', 'weighted_delta_ppl_residual_mlp', C_RM, 2.6, 0.95, 6.4, '^', '-.'),
    ],
    out_path=PERP_OUTPUT_DIR / 'perp_scale_weighted_delta_ppl.pdf',
    x_label='Perp. Scale',
    fig_size=(8.4, 5.2),
    legend_loc='upper right',
    legend_ncol=1,
    legend_fontsize=16.0,
    legend_bbox_to_anchor=(0.382, 0.555),
    x_pad_ratio=0.006,
)
